# Building NAIP mosaics with RasterFlow

This notebook demonstrates how to generate high-resolution aerial mosaics from the National Agriculture Imagery Program (NAIP) using Wherobots RasterFlow's built-in NAIP datasets. You will learn how to define an Area of Interest, build a NAIP composite mosaic, and visualize the results as RGB imagery.

It mirrors the Sentinel-2 mosaic notebook (`RasterFlow_S2_Mosaic.ipynb`), but swaps satellite imagery for NAIP's sub-meter aerial imagery.

### Built-in NAIP datasets

RasterFlow ships two built-in NAIP configurations, selectable through `DatasetEnum`. Both provide four bands — red, green, blue, and near-infrared — as 8-bit (`Byte`) values:

- **`NAIP_30CM`** — NAIP aerial imagery at **30 cm** resolution.
- **`NAIP_60CM`** — NAIP aerial imagery at **60 cm** resolution.

A built-in mosaic's bands are labeled `<dataset>:<band>` — for example `naip_30cm:red`, `naip_30cm:green`, `naip_30cm:blue`, `naip_30cm:nir`.

NAIP is flown state-by-state on a multi-year cycle, and the available resolution varies by state and year. RasterFlow selects imagery for the requested resolution and date window from its built-in NAIP index; if a location has no imagery at the requested resolution in that window, the mosaic will be empty. We use two AOIs below, each chosen because it has coverage at the target resolution:

- **30 cm → College Park, MD** (2023) — the AOI from the SAM3 notebook.
- **60 cm → Nashua, NH** (2021) — the AOI from the CHM notebook.

## Setup and Imports

In [ ]:
from datetime import datetime

import geopandas as gpd
import wkls
from rasterflow_remote import RasterflowClient
from rasterflow_remote.data_models import DatasetEnum

## Initializing the RasterFlow client

In [ ]:
rf_client = RasterflowClient()

# 30 cm NAIP mosaic — College Park, MD

We build a 30 cm NAIP mosaic over College Park, Maryland using the built-in `NAIP_30CM` dataset. College Park has 30 cm NAIP coverage in **2023**, so we use a date window covering that year. (This is the same AOI used in the SAM3 notebook, `RasterFlow_SAM3.ipynb`.)

## Preview: example mosaic

A pre-generated NAIP 30 cm mosaic over College Park is available in the Wherobots Cloud map viewer, so you can see the kind of output this section builds without waiting for the build to finish.

View the interactive map [here](https://viz.wherobots.com/?z=14.24&lat=38.99775&lng=-76.93743&l0.id=4b363cf8-20cf-464f-bbaf-820ae32d85de&l0.src=s3%3A%2F%2Fwherobots-examples%2Frasterflow%2Fmosaics%2Fcollegepark_optimized.zarr&l0.name=NAIP+30+cm+College+Park+MD&l0.clim=24.672528253600106%2C200.6443167352033&l0.mode=rgb&l0.rgb=r%2Cg%2Cb%2Cband&l0.var=variables&title=NAIP+30+cm+College+Park+MD).

> **Note:** NAIP mosaics are **4-band** (red, green, blue, and near-infrared) — this preview renders only the RGB bands, but the NIR band is present in the store for indices like NDVI or false-color views. This pre-generated sample was built via the GTI approach (see `RasterFlow_Bring_Your_Own_Rasters_NAIP.ipynb`) with its bands labeled `r` / `g` / `b` / `ir`; a mosaic you build with the built-in `NAIP_30CM` dataset labels them `red` / `green` / `blue` / `nir` instead.

## Selecting the Area of Interest (AOI)

We use College Park, Maryland — an urban/suburban area in the Northeastern US with recent 30 cm NAIP coverage.

In [ ]:
# Generate a geometry for College Park, MD using Well-Known Locations (https://github.com/wherobots/wkls)
collegepark_gdf = gpd.read_file(wkls.us.md.collegepark.geojson())

min_lon, min_lat, max_lon, max_lat = collegepark_gdf.total_bounds
print(f"Bounds: [{min_lon:.4f}, {min_lat:.4f}, {max_lon:.4f}, {max_lat:.4f}]")

## Building the 30 cm mosaic

This step builds a NAIP mosaic at 30 cm resolution using the built-in `NAIP_30CM` dataset. RasterFlow queries its built-in NAIP index for 30 cm imagery intersecting the AOI within the date window, reprojects/resamples into the target CRS, and mosaics the tiles into a Zarr store with red, green, blue, and NIR bands.

> **Note:** This step can take several minutes to complete. If you already have a pre-built mosaic store, you can skip this build by assigning its URI directly in the next cell.

In [ ]:
# Optionally skip the build by pointing at a pre-generated mosaic store:
# mosaic_store = "s3://wherobots-examples/rasterflow/mosaics/collegepark_optimized.zarr"

In [ ]:
mosaic_index = rf_client.build_mosaics(
    # Built-in NAIP 30 cm dataset (red, green, blue, nir)
    datasets=[DatasetEnum.NAIP_30CM],

    # Area of Interest (AOI)
    aoi=collegepark_gdf,

    # Acquisition window — College Park has 30 cm NAIP coverage in 2023
    start=datetime(2023, 1, 1),
    end=datetime(2024, 1, 1),

    # Output CRS (Web Mercator)
    target_crs=3857,
)

# Inspect the mosaic index (one row per output mosaic location)
mosaic_index.mosaic_index_gdf

In [ ]:
# This example AOI has one geometry, so we select the first (only) mosaic location.
mosaic_store = mosaic_index.first_row_mosaic

mosaic_store

## Build an optimized Zarr and visualize the output

RasterFlow writes its outputs as Zarr stores at native resolution, which isn't ideal for interactive viewing — every pan or zoom would have to stream full-resolution pixels. `build_zarr_multiscales` adds downsampled overview levels (image pyramids) so the map can stream coarse tiles when zoomed out and full-resolution pixels when zoomed in. This typically takes a few minutes for large outputs.

Once built, open it in the Wherobots Cloud map viewer via the store's `map_url`. (`build_zarr_multiscales` returns a `UriOutput`, whose `map_url` property links to the map viewer for that store.)

In [ ]:
optimized_store = rf_client.build_zarr_multiscales(source_store=mosaic_store)

print(f"Optimized store: {optimized_store.uri}")
# Open the freshly built mosaic in the Wherobots Cloud map viewer:
print(f"View on the map: {optimized_store.map_url}")

# 60 cm NAIP mosaic — Nashua, NH

The `NAIP_60CM` dataset builds a mosaic from NAIP's 60 cm imagery. Pick 60 cm over 30 cm when:

- You need coverage for a **year or location** that wasn't flown at 30 cm — 60 cm imagery is available for more states and more years.
- You're mosaicking a **large area** and want smaller outputs and faster builds — 60 cm imagery is roughly a quarter of the pixels of 30 cm.

We build over Nashua, NH, which has a mix of urban, park, and forest cover. New Hampshire has 60 cm NAIP coverage in **2021**, so we use a date window covering that year. (This is the same AOI used in the CHM notebook, `RasterFlow_CHM.ipynb`.)

## Preview: example mosaic

A pre-generated 60 cm NAIP mosaic over Nashua, NH is available in the Wherobots Cloud map viewer.

View the interactive map [here](https://viz.wherobots.com/?z=11&lat=42.7654&lng=-71.4676&l0.id=4a8d3619-291c-4acb-8246-99aea96c3bc4&l0.src=s3%3A%2F%2Fwherobots-examples%2Frasterflow%2Fmosaics%2Fnashua_optimized.zarr&l0.name=NAIP+60+cm+Nashua+NH&l0.clim=33.370882202482704%2C201.81320110343603&l0.mode=rgb&l0.rgb=red%2Cgreen%2Cblue%2Cband&l0.var=variables&title=NAIP+60+cm+Nashua+NH).

## Selecting the AOI and building the mosaic

We build the Nashua mosaic with `NAIP_60CM` the same way as the 30 cm mosaic above. As before, you can pass the resulting store to `build_zarr_multiscales` and open its `map_url` in the Wherobots Cloud map viewer to view your freshly built output.

In [ ]:
# Generate a geometry for Nashua, NH using Well-Known Locations (https://github.com/wherobots/wkls)
nashua_gdf = gpd.read_file(wkls.us.nh.nashua.geojson())

coarse_index = rf_client.build_mosaics(
    # Built-in NAIP 60 cm dataset (red, green, blue, nir)
    datasets=[DatasetEnum.NAIP_60CM],

    # Area of Interest (AOI)
    aoi=nashua_gdf,

    # Acquisition window — New Hampshire has 60 cm NAIP coverage in 2021
    start=datetime(2021, 1, 1),
    end=datetime(2022, 1, 1),

    # Output CRS (Web Mercator)
    target_crs=3857,
)

coarse_store = coarse_index.first_row_mosaic
print(f"60 cm mosaic store: {coarse_store}")

## Summary

This notebook demonstrated how to:

1. Build a 30 cm NAIP aerial mosaic over College Park, MD with the built-in `DatasetEnum.NAIP_30CM` dataset
2. Optimize the Zarr store with overviews and open it in the Wherobots Cloud map viewer
3. Build a 60 cm NAIP mosaic over Nashua, NH with `DatasetEnum.NAIP_60CM`

### Next steps

- Compare a NAIP mosaic with the Sentinel-2 mosaic over the same AOI (see `RasterFlow_S2_Mosaic.ipynb`)
- Run a model on the NAIP mosaic — for example text-prompted object detection with SAM3 on the College Park mosaic (`RasterFlow_SAM3.ipynb`) or tree canopy height with Meta CHM on the Nashua mosaic (`RasterFlow_CHM.ipynb`)
- Use the mosaic as input for **model inference** using [predict_mosaic](https://docs.wherobots.com/reference/rasterflow/client#predict_mosaic)